# S8 multi-doc 질의 파일럿

생성 지시 v1\~v4 파일럿의 수율, 불통과 사유, 비용을 보고 본 생성(G3, 300\~500건)의 규모와 판정 규칙을 정하는 근거 자료다(`docs/slices/08-multidoc-queries.md`).

- 저장된 검사 기록(`data/multidoc/pilot_v1/check`, `pilot_v2/check`, `check`)만 읽는다. API를 호출하지 않는다
- 현재 설정은 `conf` order 0\~9 v3·10\~19 v4, `law` 0\~9 v3, `questioner` 0\~9 v4다(`check`). 검사 지시는 v1이 `check_v1`, 나머지가 `check_v2`
- 원문 인용은 짧게 출력한다
- 판정은 저장된 검사 응답으로 다시 계산한다. `old`는 이전 규칙 1(생성·검사 쪽 인용이 하나라도 원문에 없으면 `quote_missing`), `new`는 2026-10-06 채택한 규칙 1(검사 쪽 인용이 원문에 없으면 그 근거 항목만 빼고 판정)이다. 1\~8절은 정확 대조로, 9절은 그 뒤 채택한 유사도 대조로 판정한다. 1\~7절의 비율은 `old`로 보고 결정 근거로 쓴 값이고, 6절 외삽은 두 규칙을 함께 낸다

In [1]:
import json
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from rapidfuzz import fuzz

from rag.index import load_cfg
from rag.multidoc.check import found, judge, norm, quoted, to_doc_ids

In [2]:
ROOT = Path.cwd().parent
cfg = load_cfg()
M = cfg.multidoc
TYPES = list(M.types)
DOCS = ROOT / cfg.paths.multidoc_docs
RUNS = {"v1": ROOT / "data/multidoc/pilot_v1/check", "v2": ROOT / "data/multidoc/pilot_v2/check",
        "v3·v4": ROOT / cfg.paths.multidoc_check}
TARGETS = [300, 400, 500]  # 본 생성 목표 질의 수(PLAN G3)
AVAILABLE = {"conf": 2066, "law": 385, "questioner": 4095}  # S7 가용 후보 집합 수(DECISIONS D-13)
SONNET = ROOT / "_workspace/s08_sonnet_check/out"  # 검사 모델 비교용 Sonnet 서브에이전트 검사 결과
N_EXAMPLES = 2  # 출력할 통과·불통과 예시 수
EXACT = SimpleNamespace(min_ratio=1.0, min_chars=0)  # 1~8절: 결정 당시의 정확 대조
dict(M.gen), dict(M.check)

({'n_per_type': 10,
  'overview_chars': 300,
  'max_gold': 5,
  'prompt_version': {'conf': 'gen_v4', 'law': 'gen_v3', 'questioner': 'gen_v4'}},
 {'base_url': 'https://openrouter.ai/api/v1',
  'api_key_env': 'OPENROUTER_API',
  'model': 'openai/gpt-6-luna',
  'provider': 'openai',
  'seed': 20260929,
  'reasoning_effort': 'medium',
  'max_tokens': 16000,
  'prompt_version': 'check_v2',
  'quote_match': {'min_ratio': 0.8, 'min_chars': 15}})

In [3]:
def context(doc_id):
    return (DOCS / f"{doc_id}.txt").read_text(encoding="utf-8")


pools = {json.loads(line)["pool_id"]: json.loads(line)
         for line in (ROOT / cfg.paths.multidoc_pools).read_text(encoding="utf-8").splitlines()}


def judge_old(rec, elements, answerable, new):
    # 이전 규칙 1: 검사 쪽 인용도 원문에 없으면 quote_missing이고, 근거 항목은 빼지 않는다.
    # 생성 쪽 검사(gen_invalid, 생성 인용)는 새 판정(new)에서 가져온다
    quotes = [(s["doc_id"], s["quote"]) for el in elements for s in el["support"]]
    reasons = [x for x in new["reasons"] if x in ("gen_invalid", "quote_missing")]
    if "quote_missing" not in reasons and any(d is None or not quoted(q, context(d))
                                              for d, q in quotes):
        reasons.append("quote_missing")
    if not answerable:
        reasons.append("unanswerable")
    supports = [{s["doc_id"] for s in el["support"] if s["doc_id"]} for el in elements]
    gold = sorted(set().union(*supports)) if supports else []
    unique = {next(iter(x)) for x in supports if len(x) == 1}
    if len(gold) < 2:
        reasons.append("single_doc")
    if set(gold) - unique:
        reasons.append("substitutable")
    if set(gold) != set(rec["gen"]["seed_doc_ids"]):
        reasons.append("seed_mismatch")
    return {"passed": not reasons, "reasons": reasons, "gold_doc_ids": gold}


def verdicts(rec, elements, answerable, match=None):
    if rec["gen"]["status"] != "ok":
        skip = {"passed": False, "reasons": ["gen_skip"], "gold_doc_ids": []}
        return skip, skip
    new = judge(rec["gen"], elements, answerable, rec["pool_doc_ids"],
                {d: context(d) for d in rec["pool_doc_ids"]}, M.gen.max_gold,
                pools[rec["pool_id"]]["seed_doc_ids"], match=match or EXACT)
    return judge_old(rec, elements, answerable, new), new


def load_run(name, path):
    rows = []
    for f in sorted(path.glob("*.json")):
        r = json.loads(f.read_text(encoding="utf-8"))
        answerable = json.loads(r["response"])["answerable"] if r.get("response") else False
        v, v_new = verdicts(r, r.get("elements") or [], answerable)
        g, u = r["gen"], r.get("usage") or {}
        rows.append({"run": name, "pool_id": r["pool_id"], "type": r["type"],
                     "gen_version": g.get("prompt_version"), "status": g["status"],
                     "query_form": g.get("query_form"), "passed": v["passed"],
                     "reasons": v["reasons"], "passed_new": v_new["passed"],
                     "reasons_new": v_new["reasons"],
                     "n_seed": len(g.get("seed_doc_ids") or []),
                     "n_gold": len(v["gold_doc_ids"]), "in_tok": u.get("prompt_tokens", 0),
                     "out_tok": u.get("completion_tokens", 0), "cost": u.get("cost", 0.0),
                     "rec": r})
    return rows

## 1. 검사 기록

In [4]:
df = pd.DataFrame([row for name, p in RUNS.items() for row in load_run(name, p)])
print(df.shape)
df.groupby(["run", "type", "gen_version"]).size()

(100, 16)


run    type        gen_version
v1     conf        gen_v1         10
       law         gen_v1         10
       questioner  gen_v1         10
v2     conf        gen_v2         10
       law         gen_v2         10
       questioner  gen_v2         10
v3·v4  conf        gen_v3         10
                   gen_v4         10
       law         gen_v3         10
       questioner  gen_v4         10
dtype: int64

v1·v2 기록은 이전 규칙으로 판정했으므로, 다시 계산한 `old`가 기록과 같은지 확인한다.

In [5]:
stored = df["rec"].apply(lambda r: r["verdict"]["passed"])
df[df["run"] != "v3·v4"].assign(stored=stored).pipe(lambda d: (d["passed"] == d["stored"]).all())

np.True_

## 2. 버전별 수율

후보 집합 대비 생성 `ok` 비율과 통과 비율. 파일럿 표본이 유형마다 10\~20개라 비율 차이 10%p는 후보 집합 1\~2개 차이다.

In [6]:
yield_ = df.groupby(["run", "type"]).agg(pools=("pool_id", "size"),
                                         gen_ok=("status", lambda s: (s == "ok").sum()),
                                         passed=("passed", "sum"))
yield_["pass_rate"] = (yield_["passed"] / yield_["pools"]).round(3)
total = df.groupby("run").agg(pools=("pool_id", "size"),
                              gen_ok=("status", lambda s: (s == "ok").sum()),
                              passed=("passed", "sum"))
total["pass_rate"] = (total["passed"] / total["pools"]).round(3)
display(yield_)
total

pools  gen_ok  passed  pass_rate
run   type                                        
v1    conf           10       6       1        0.1
      law            10       5       4        0.4
      questioner     10       3       0        0.0
v2    conf           10       3       3        0.3
      law            10       9       5        0.5
      questioner     10       4       0        0.0
v3·v4 conf           20      16       4        0.2
      law            10      10       4        0.4
      questioner     10       5       3        0.3

,pools,gen_ok,passed,pass_rate
run,,,,
v1,30,14,5,0.167
v2,30,16,8,0.267
v3·v4,40,31,11,0.275


In [7]:
cur = df[df["run"] == "v3·v4"].copy()
cur.groupby(["type", "gen_version"]).agg(pools=("pool_id", "size"),
                                          gen_ok=("status", lambda s: (s == "ok").sum()),
                                          passed=("passed", "sum"))

pools  gen_ok  passed
type       gen_version                       
conf       gen_v3          10       6       3
           gen_v4          10      10       1
law        gen_v3          10      10       4
questioner gen_v4          10       5       3

## 3. 불통과 사유 (v3·v4)

한 질의에 사유가 여러 개 붙는다. `단독`은 그 사유 하나로만 탈락한 건수다.

In [8]:
fail = cur[(cur["status"] == "ok") & ~cur["passed"]]
reasons = fail.explode("reasons")
counts = reasons.groupby(["reasons", "type"]).size().unstack(fill_value=0)
counts["단독"] = (fail[fail["reasons"].str.len() == 1].explode("reasons")
                  .groupby("reasons").size()).reindex(counts.index, fill_value=0)
print("생성 ok 중 불통과", len(fail), "/", (cur["status"] == "ok").sum())
counts

생성 ok 중 불통과 20 / 31


type,conf,law,questioner,단독
reasons,,,,
quote_missing,5,4,2,4
seed_mismatch,7,3,0,2
substitutable,8,3,1,3
unanswerable,1,0,0,1


### 3.1 `quote_missing`의 출처

규칙 1은 생성 쪽 `evidence`와 검사 쪽 `support`의 인용을 모두 원문과 대조한다. 어느 쪽 인용이 원문에 없는지 센다.

In [9]:
def quote_misses(rec):
    gen_q = [(e["doc_id"], e["quote"]) for e in rec["gen"]["evidence"]]
    chk_q = [(s["doc_id"], s["quote"]) for el in rec["elements"] for s in el["support"]]
    gen_miss = [q for d, q in gen_q if not quoted(q, context(d))]
    chk_miss = [q for d, q in chk_q if d is None or not quoted(q, context(d))]
    return len(gen_q), len(gen_miss), len(chk_q), chk_miss


qm = []
for _, row in fail[fail["reasons"].apply(lambda r: "quote_missing" in r)].iterrows():
    n_gen, gen_miss, n_chk, chk_miss = quote_misses(row["rec"])
    qm.append({"pool_id": row["pool_id"], "gen_quotes": n_gen, "gen_miss": gen_miss,
               "chk_quotes": n_chk, "chk_miss": len(chk_miss), "reasons": row["reasons"],
               "example": chk_miss[0][:60] if chk_miss else ""})
pd.DataFrame(qm)

,pool_id,gen_quotes,gen_miss,chk_quotes,chk_miss,reasons,example
0,conf-0005,2,0,12,1,"[quote_missing, substitutable, seed_mismatch]",그러니까 지금 이렇게 논의가 헷갈리는 게 전임자랑 근로시간면제자랑 구분을 하고 있잖...
1,conf-0013,6,0,18,2,"[quote_missing, substitutable, seed_mismatch]",위원장님은 제가 말씀을 드려서 모셨고요.
2,conf-0014,3,0,20,1,"[quote_missing, seed_mismatch]",그런데 이 업체가 영세하단 말이에요.
3,conf-0016,4,0,20,1,"[quote_missing, substitutable, seed_mismatch]",그러면 우선 중간 단계로서 ‘공공보건의료시설 등 대통령령이 정하는 공공의 이익을 위...
4,conf-0019,4,0,10,1,[quote_missing],"아니, 그러니까 기존 국방망의 성능을 개선한다든가 개량해 가지고 업그레이드시켜면 될..."
5,law-0002,4,0,16,2,[quote_missing],그래서 이 부분은 소득 자체를 신고하는 것이 아니라 소득 발생 원인이 되는 거래행위...
6,law-0007,4,0,18,2,"[quote_missing, substitutable, seed_mismatch]",그런데 행정행위로 이루어지는 톱다운 방식에 따라서 실링이라고 하는 지출한도를 주는데...
7,law-0008,2,0,14,1,[quote_missing],그런데 이것은 세무조사는 아니거든요. 스스로 납세자들이 종합소득세를 신고하는 과정에...
8,law-0009,3,0,12,1,"[quote_missing, substitutable]",그러면 계도 기간을 두는 것은 어때요? 예를 들어서 한 2년이나 이렇게 계도 기간을...
9,questioner-0005,5,0,16,1,"[quote_missing, substitutable]",그래서 그런 연령들을 생각해서라도 이번에 최소한 2∼3년의 동결을 끝내고 내년 예산...


생성 쪽 인용은 모두 원문에 있다. 검사 쪽은 질의마다 인용 10\~20개 중 1\~2개가 원문과 다르다. 아래는 원문에서 가장 비슷한 대목과 나란히 본 예시다.

In [10]:
def nearest(quote, text, width=None):
    width = width or len(quote)
    head = quote[:8]
    i = text.find(head)
    return text[i:i + width] if i >= 0 else "(앞 8자도 원문에 없음)"


for row in qm[:3]:
    rec = cur.set_index("pool_id").loc[row["pool_id"], "rec"]
    for el in rec["elements"]:
        for s in el["support"]:
            if s["doc_id"] and not quoted(s["quote"], context(s["doc_id"])):
                print(row["pool_id"], "| 검사:", s["quote"][:80])
                near = nearest(s["quote"], context(s["doc_id"]))
                print(" " * len(row["pool_id"]), "| 원문:", near[:80])

conf-0005 | 검사: 그러니까 지금 이렇게 논의가 헷갈리는 게 전임자랑 근로시간면제자랑 구분을 하고 있잖아요, 그렇지요?
          | 원문: (앞 8자도 원문에 없음)
conf-0013 | 검사: 위원장님은 제가 말씀을 드려서 모셨고요.
          | 원문: 위원장님은 제가 모셨고 위원장님께서 두 
conf-0013 | 검사: 7월 14일 날 1차 발표하고, 7월 28일 날, 14일 만에 또다시 2차 권고안 100% 수용했습니다.
          | 원문: 7월 14일 날 1차 권고안을 내는데 어마어마한 권고안을 냅니다.  ‘인권침해사건진상조사위원회 설치하라’
conf-0014 | 검사: 그런데 이 업체가 영세하단 말이에요.
          | 원문: 그런데 이 업체가 아무리 팔아 봐도 


### 3.2 `seed_mismatch`의 차이 유형

검사 쪽 정답 집합이 생성 쪽 `seed_doc_ids`보다 문서를 더했는지(+), 뺐는지(−) 센다.

In [11]:
def seed_diff(rec):
    seed, gold = set(rec["gen"]["seed_doc_ids"]), set(rec["verdict"]["gold_doc_ids"])
    return len(gold - seed), len(seed - gold)


sm = fail[fail["reasons"].apply(lambda r: "seed_mismatch" in r)].copy()
sm[["added", "removed"]] = sm["rec"].apply(lambda r: pd.Series(seed_diff(r)))
sm["kind"] = sm.apply(lambda r: "+" if r["removed"] == 0 else ("−" if r["added"] == 0 else "+−"),
                      axis=1)
display(sm.groupby(["type", "kind"]).size())
sm[["pool_id", "n_seed", "n_gold", "added", "removed", "reasons"]]

type  kind
conf  +       5
      +−      1
      −       1
law   +       2
      +−      1
dtype: int64

,pool_id,n_seed,n_gold,added,removed,reasons
63,conf-0003,3,2,0,1,[seed_mismatch]
65,conf-0005,2,3,1,0,"[quote_missing, substitutable, seed_mismatch]"
70,conf-0010,3,4,1,0,"[substitutable, seed_mismatch]"
72,conf-0012,2,3,1,0,"[substitutable, seed_mismatch]"
73,conf-0013,3,6,3,0,"[quote_missing, substitutable, seed_mismatch]"
74,conf-0014,3,4,1,0,"[quote_missing, seed_mismatch]"
76,conf-0016,4,7,4,1,"[quote_missing, substitutable, seed_mismatch]"
83,law-0003,2,7,5,0,"[substitutable, seed_mismatch]"
86,law-0006,2,3,1,0,[seed_mismatch]
87,law-0007,4,8,5,1,"[quote_missing, substitutable, seed_mismatch]"


### 3.3 `substitutable`와 정답 문서 수

규칙 4는 정답 문서마다 그 문서에만 있는 요소를 요구한다. 생성 쪽 정답 수(`n_seed`)별로 이 사유가 붙은 비율을 본다.

In [12]:
ok = cur[cur["status"] == "ok"].copy()
ok["substitutable"] = ok["reasons"].apply(lambda r: "substitutable" in r)
ok.groupby(["type", "n_seed"]).agg(n=("pool_id", "size"), substitutable=("substitutable", "sum"),
                                   passed=("passed", "sum"))

n  substitutable  passed
type       n_seed                          
conf       2       7              2       4
           3       5              2       0
           4       2              2       0
           5       2              2       0
law        2       7              1       4
           3       1              1       0
           4       2              1       0
questioner 2       2              0       1
           3       2              0       2
           5       1              1       0

## 4. 정답 문서 수 분포 (통과, 전체 버전)

In [13]:
df[df["passed"]].groupby(["type", "n_gold"]).size().unstack(fill_value=0)

n_gold,2,3
type,,
conf,8,0
law,13,0
questioner,1,2


## 5. 검사 쪽 인용 오류를 빼면

규칙 1의 대상에서 검사 쪽 인용만 바꿔 다시 판정한다. 원문에 없는 검사 쪽 인용을 그 `support` 항목째 빼고(그 문서가 그 요소를 담는다는 근거가 없으므로) 규칙 1\~5를 그대로 적용한다. 생성 쪽 인용은 그대로 대조한다. 이 규칙은 2026-10-06 채택해 `rag.multidoc.check`의 판정이 됐다(`new`).

In [14]:
alt = ok[["pool_id", "type", "passed", "passed_new", "reasons_new"]].rename(
    columns={"passed_new": "passed_alt", "reasons_new": "reasons_alt"})
display(alt.groupby("type")[["passed", "passed_alt"]].sum())
alt[alt["passed"] != alt["passed_alt"]]

,passed,passed_alt
type,,
conf,4,5
law,4,6
questioner,3,4


,pool_id,type,passed,passed_alt,reasons_alt
79,conf-0019,conf,False,True,[]
82,law-0002,law,False,True,[]
88,law-0008,law,False,True,[]
96,questioner-0006,questioner,False,True,[]


## 6. 본 생성 외삽

현재 설정(v3·v4) 유형별 통과율로 목표 질의 수에 필요한 후보 집합 수를 낸다. 유형별 후보 집합 수는 같게 두고 계산한 뒤, 채택한 설정(conf·questioner 350, law 250)으로 다시 낸다. 비율 추정은 유형마다 10\~20개 표본이라 폭이 넓다.

In [15]:
rate = cur.groupby("type").agg(pools=("pool_id", "size"), passed=("passed", "sum"),
                               gen_ok=("status", lambda s: (s == "ok").sum()))
rate["pass_rate"] = rate["passed"] / rate["pools"]
rate["ok_rate"] = rate["gen_ok"] / rate["pools"]
alt_rate = alt.groupby("type")["passed_alt"].sum() / rate["pools"]
rate["pass_rate_alt"] = alt_rate
rate["available"] = pd.Series(AVAILABLE)
rate.round(3)

,pools,passed,gen_ok,pass_rate,ok_rate,pass_rate_alt,available
type,,,,,,,
conf,20,4,16,0.2,0.8,0.25,2066
law,10,4,10,0.4,1.0,0.60,385
questioner,10,3,5,0.3,0.5,0.40,4095


In [16]:
def expected(n_per_type, col):
    per = rate[col] * rate["available"].clip(upper=n_per_type)
    return per.round(0).astype(int).to_dict() | {"합": int(per.sum().round())}


rows = []
for n in [250, 385, 500, 600]:
    rows.append({"n_per_type": n, "이전 규칙": expected(n, "pass_rate"),
                 "새 규칙": expected(n, "pass_rate_alt")})
pd.DataFrame(rows)

,n_per_type,이전 규칙,새 규칙
0,250,"{'conf': 50, 'law': 100, 'questioner': 75, '합'...","{'conf': 62, 'law': 150, 'questioner': 100, '합..."
1,385,"{'conf': 77, 'law': 154, 'questioner': 116, '합...","{'conf': 96, 'law': 231, 'questioner': 154, '합..."
2,500,"{'conf': 100, 'law': 154, 'questioner': 150, '...","{'conf': 125, 'law': 231, 'questioner': 200, '..."
3,600,"{'conf': 120, 'law': 154, 'questioner': 180, '...","{'conf': 150, 'law': 231, 'questioner': 240, '..."


`law`는 가용 후보 집합이 385개라 그 이상 늘지 않는다. 아래는 목표마다 필요한 유형별 후보 집합 수(law 상한 반영)다.

In [17]:
def needed(target, col):
    for n in range(250, 4096, 10):
        if expected(n, col)["합"] >= target:
            return n
    return None


pd.DataFrame([{"target": t, "이전 규칙": needed(t, "pass_rate"),
               "새 규칙": needed(t, "pass_rate_alt")} for t in TARGETS])

,target,이전 규칙,새 규칙
0,300,340,250
1,400,500,320
2,500,700,420


채택한 설정(conf·questioner 350, law 250)과 새 규칙의 기대 통과 수.

In [18]:
ADOPTED = dict(M.n_pools_per_type)
per = (rate["pass_rate_alt"] * pd.Series(ADOPTED)).round(0).astype(int)
print(ADOPTED, per.to_dict(), "합", int(per.sum()))

{'conf': 350, 'law': 250, 'questioner': 350} {'conf': 88, 'law': 150, 'questioner': 140} 합 378


### 6.1 비용과 시간

검사 비용은 OpenRouter 응답 `usage.cost` 실측이다. 생성 `skip`은 검사를 호출하지 않는다. 생성 쪽은 Claude Code 에이전트라 API 과금이 없고, 파일럿 실측 소요는 후보 집합 10개에 6.5\~19분(에이전트 하나)이다.

In [19]:
calls = cur[cur["status"] == "ok"]
per_call = calls["cost"].mean()
print(f"검사 {len(calls)}건 비용 합 ${calls['cost'].sum():.4f}, 호출당 ${per_call:.4f}, "
      f"입력 {calls['in_tok'].sum():,}·출력 {calls['out_tok'].sum():,} 토큰")
print(f"전체 버전 검사 비용 합 ${df['cost'].sum():.4f}")

ok_rate = (cur["status"] == "ok").mean()
est = []
for n in [250, 385, 500, 600]:
    n_pools = int(rate["available"].clip(upper=n).sum())
    est.append({"n_per_type": n, "후보 집합": n_pools, "검사 호출(추정)": round(n_pools * ok_rate),
                "검사 비용 $": round(n_pools * ok_rate * per_call, 2),
                "생성 에이전트 호출(10개씩)": -(-n_pools // 10)})
pd.DataFrame(est)

검사 31건 비용 합 $0.1044, 호출당 $0.0034, 입력 467,394·출력 91,967 토큰
전체 버전 검사 비용 합 $0.1890


,n_per_type,후보 집합,검사 호출(추정),검사 비용 $,생성 에이전트 호출(10개씩)
0,250,750,581,1.96,75
1,385,1155,895,3.01,116
2,500,1385,1073,3.62,139
3,600,1585,1228,4.14,159


## 7. 예시 (v3·v4)

In [20]:
def show(row):
    rec = row["rec"]
    print(f"[{row['pool_id']} {row['gen_version']} {row['query_form']}] 통과={row['passed']} "
          f"{row['reasons']}")
    print("  질의:", rec["gen"]["query"])
    for el in rec["elements"][:4]:
        print("  -", el["fact"][:80], "| 문서", len({s["doc_id"] for s in el["support"]}))


for t in TYPES:
    sub = cur[(cur["type"] == t) & (cur["status"] == "ok")]
    for _, row in pd.concat([sub[sub["passed"]].head(1), sub[~sub["passed"]].head(1)]).iterrows():
        show(row)
    print()

[conf-0002 gen_v3 nan] 통과=True []
  질의: 2017년 9월 20일 정무위원회 법안심사소위원회에서 자본시장과 금융투자업에 관한 법률 개정안 심사 중 시장질서 교란행위에 형벌을 도입하는 문제에 대해 위원과 정부는 어떤 입장을 보였나?
  - 김종석 위원은 시장질서 교란행위가 불공정거래행위보다 경한 위반인데 형벌과 과징금을 함께 부과하면 불공정거래행위보다 제재가 더 무거워져 비례의 원 | 문서 1
  - 정부는 시장질서 교란행위에 형벌을 도입하는 데 부정적이었다. 과징금은 형벌을 적용하기 어려운 행위도 제재할 필요가 있어 도입된 것이고, 구성요건 | 문서 1
  - 정부는 현재 형벌만 부과되는 내부자거래·시세조종·부정거래에 과징금도 추가하는 방안을 검토하고 있다고 밝혔다. | 문서 1
[conf-0003 gen_v3 nan] 통과=False ['seed_mismatch']
  질의: 2020년 12월 30일 법제사법위원회 법안심사제1소위원회에서 중대재해 처벌 법안 심사 중 경영책임자의 의무 위반과 중대산업재해 사이의 인과관계 추정 조항에 대해 위원들은 어떤 입장을 보였나?
  - 박주민 위원은 인과관계 추정 조항이 실제로 별다른 도움이 되지 않을 수 있고, 조항이 없어도 판단할 수 있다는 취지에 동의했다. | 문서 1
  - 송기헌 위원은 경영책임자의 의무 위반과 재해 사이의 문제를 단순한 인과관계 추정으로 다루는 것은 맞지 않는다고 보았다. 의무 위반에서 구체적인  | 문서 1

[law-0000 gen_v3 nan] 통과=True []
  질의: 방송법 일부개정법률안 심사에서 2007년 9월 11일 방송통신특별위원회와 2011년 12월 22일 문화체육관광방송통신위원회 법안심사소위원회에서 각각 논의된 쟁점은 무엇인가?
  - 2007년 9월 11일에는 IPTV와의 경쟁을 이유로 플랫폼사업자·PP 간 소유겸영 및 수직결합 규제를 완화하는 것이 콘텐츠 다양성을 해치거나  | 문서 1
  - 2011년 12월 22일에는 시청자미디어센터를 전

## 8. 검사 모델 비교: luna와 Sonnet

luna가 생성 쪽(Sonnet)보다 작은 모델이라 불통과가 늘었는지 보려고, v3·v4 생성 ok 31건을 Sonnet 서브에이전트에 같은 입력(`check_v2` 메시지, 생성 쪽 시작 묶음·답·근거 없음)으로 검사시켰다(사용자 요청 2026-10-06). 결과는 `_workspace/s08_sonnet_check/out/`에 있고, 같은 판정 코드(`new`)로 비교한다. Sonnet은 생성과 같은 계열이라 이 결과는 진단용이고 판정에 쓰지 않는다.

- 실행 중 오류: conf-0012는 첫 에이전트가 다른 후보 집합 입력과 같다고 오판해 다시 검사했다. conf-0002는 에이전트가 문서 3\~14를 끝까지 읽지 못하고 키워드 검색으로 대신했다

In [21]:
cmp = []
for _, row in ok.iterrows():
    rec = row["rec"]
    out = json.loads((SONNET / f"{row['pool_id']}.json").read_text(encoding="utf-8"))
    s_el = to_doc_ids(out["elements"], rec["pool_doc_ids"])
    _, s_new = verdicts(rec, s_el, out["answerable"])
    _, l_new = verdicts(rec, rec["elements"], json.loads(rec["response"])["answerable"])
    cmp.append({"pool_id": row["pool_id"], "type": row["type"], "luna": l_new["passed"],
                "sonnet": s_new["passed"], "luna_reasons": l_new["reasons"],
                "sonnet_reasons": s_new["reasons"],
                "luna_quotes": sum(len(e["support"]) for e in rec["elements"]),
                "sonnet_quotes": sum(len(e["support"]) for e in s_el),
                "same_gold": set(l_new["gold_doc_ids"]) == set(s_new["gold_doc_ids"])})
cmp = pd.DataFrame(cmp)
display(pd.crosstab(cmp["luna"], cmp["sonnet"]))
cmp[["luna_quotes", "sonnet_quotes"]].sum()

sonnet,False,True
luna,,
False,11,5
True,0,15


luna_quotes      413
sonnet_quotes    238
dtype: int64

사유별 건수. `seed_mismatch`는 두 검사가 함께 내고(생성 쪽 누락), 차이는 주로 `substitutable`이다.

In [22]:
reason_counts = {m: cmp[f"{m}_reasons"].explode().value_counts() for m in ["luna", "sonnet"]}
pd.DataFrame(reason_counts).fillna(0)

,luna,sonnet
seed_mismatch,10,9.0
substitutable,12,5.0
unanswerable,1,0.0


In [23]:
cmp[cmp["luna"] != cmp["sonnet"]][["pool_id", "luna_reasons", "sonnet_reasons", "same_gold"]]

,pool_id,luna_reasons,sonnet_reasons,same_gold
3,conf-0005,"[substitutable, seed_mismatch]",[],False
7,conf-0011,[unanswerable],[],True
10,conf-0014,[seed_mismatch],[],False
14,conf-0018,[substitutable],[],True
29,questioner-0005,[substitutable],[],True


## 9. 인용 유사도 대조 (PR #30 뒤 결정)

1\~8절은 결정 당시의 정확 대조(`EXACT`)로 판정했다. 검사 쪽 인용 중 원문과 글자가 다른 것을 원문에서 가장 잘 맞는 부분 문자열(rapidfuzz `partial_ratio`)과 맞춰 보면, 지어낸 인용은 없고 접속어·존칭·낱말을 바꿔 옮긴 것이었다. 그래서 `quote_match.min_chars`자 이상 인용은 문자 유사도가 `quote_match.min_ratio` 이상이면 원문에 있는 것으로 본다. 인정한 인용의 문구는 원문에 맞춰 고치지 않는다(PR #31). 짧은 인용은 정확 대조만 한다. 인정하지 못한 근거에 판정이 좌우되면 `quote_dependent`로 불통과한다(D-14).

이 절의 문자 유사도 판정은 S8b에서 Jev 의미 판정으로 바뀌었다(`notebooks/08_02_인용의미.ipynb`, D-14).

In [24]:
def similarity(quote, ctx):
    return fuzz.partial_ratio(norm(quote), norm(ctx)) / 100


def window(quote, ctx):
    c = norm(ctx)
    a = fuzz.partial_ratio_alignment(norm(quote), c)
    return c[a.dest_start:a.dest_end]


bad = []
for _, row in ok.iterrows():
    for el in row["rec"]["elements"]:
        for s in el["support"]:
            if s["doc_id"] and not quoted(s["quote"], context(s["doc_id"])):
                ctx = context(s["doc_id"])
                bad.append({"pool_id": row["pool_id"], "chars": len(norm(s["quote"])),
                            "ratio": round(similarity(s["quote"], ctx), 2),
                            "check": norm(s["quote"])[:50],
                            "source": window(s["quote"], ctx)[:50]})
bad = pd.DataFrame(bad)
print(len(bad), "개, 길이", bad["chars"].min(), "자 이상, 유사도", bad["ratio"].min(), "~",
      bad["ratio"].max())
bad

14 개, 길이 20 자 이상, 유사도 0.86 ~ 0.99


,pool_id,chars,ratio,check,source
0,conf-0005,55,0.93,그러니까 지금 이렇게 논의가 헷갈리는 게 전임자랑 근로시간면제자랑 구분을 하고 있잖...,"? 지금 이렇게 논의가 헷갈리는 게 전임자랑 근로시간면제자랑 구분을 하고 있잖아요,..."
1,conf-0013,22,0.95,위원장님은 제가 말씀을 드려서 모셨고요.,위원장은 제가 말씀을 드려서 모셨고요.
2,conf-0013,58,0.97,"7월 14일 날 1차 발표하고, 7월 28일 날, 14일 만에 또다시 2차 권고안 ...","14일 날 1차 발표하고, 7월 28일 날, 14일 만에 또다시 2차 권고안 10..."
3,conf-0014,20,0.90,그런데 이 업체가 영세하단 말이에요.,데 이 업체가 영세하단 말이에요. 그
4,conf-0016,163,0.98,그러면 우선 중간 단계로서 ‘공공보건의료시설 등 대통령령이 정하는 공공의 이익을 위...,우선 중간 단계로서 ‘공공보건의료시설 등 대통령령이 정하는 공공의 이익을 위한 시...
5,conf-0019,101,0.97,"아니, 그러니까 기존 국방망의 성능을 개선한다든가 개량해 가지고 업그레이드시켜면 될...","쎄요, 그러니까 기존 국방망의 성능을 개선한다든가 개량해 가지고 업그레이드시키면 될..."
6,law-0002,100,0.97,그래서 이 부분은 소득 자체를 신고하는 것이 아니라 소득 발생 원인이 되는 거래행위...,그래서 이것은 소득 자체를 신고하는 것이 아니라 소득 발생 원인이 되는 거래행위를 ...
7,law-0002,101,0.98,사회임대주택이라는 것은 상대적으로 일반적인 민간임대주택보다 조금 더 소득 계층이 낮...,사회적 임대주택이라는 것은 상대적으로 일반적인 민간임대주택보다 조금 더 소득 계층이...
8,law-0007,142,0.99,그런데 행정행위로 이루어지는 톱다운 방식에 따라서 실링이라고 하는 지출한도를 주는데...,그래서 행정행위로 이루어지는 톱다운 방식에 따라서 실링이라고 하는 지출한도를 주는데...
9,law-0007,111,0.95,기획재정부장관이 관리하는 기금뿐만 아니라 국가가 직접 관리하지 않는 기금까지 다 포...,중앙관서의 장이 관리하는 기금뿐만 아니라 국가가 직접 관리하지 않는 기금까지 다 포...


기준값 근거: 원문에 그대로 있는 검사 쪽 인용을 같은 후보 집합의 다른 문서(그 인용이 없는 문서)에 대조한 유사도. 검사가 같은 쟁점의 다른 문서에 인용을 잘못 붙이는 경우를 흉내 낸다. 수 분 걸린다.

In [25]:
neg = []
for _, row in ok.iterrows():
    rec = row["rec"]
    for el in rec["elements"]:
        for s in el["support"]:
            if not (s["doc_id"] and quoted(s["quote"], context(s["doc_id"]))):
                continue
            for d in rec["pool_doc_ids"]:
                if d != s["doc_id"] and not quoted(s["quote"], context(d)):
                    neg.append({"chars": len(norm(s["quote"])),
                                "ratio": similarity(s["quote"], context(d)),
                                "quote": norm(s["quote"])[:40],
                                "found": window(s["quote"], context(d))[:40]})
neg = pd.DataFrame(neg).sort_values("ratio", ascending=False)
print(len(neg), "쌍", {t: int((neg["ratio"] >= t).sum()) for t in [0.6, 0.7, 0.8, 0.86]})
neg.head(8)

4285 쌍 {0.6: 37, 0.7: 6, 0.8: 1, 0.86: 0}


,chars,ratio,quote,found
2170,38,0.815789,"권순일, 박영수, 곽상도, 김수남, 최재경, 그리고 홍 모 씨입니다.","주셨습니다. 권순일, 박영수, 곽상도, 김수남, 최재경, 홍 모 씨."
4250,9,0.777778,이유가 있습니까?,개가 있습니까?
4252,9,0.777778,이유가 있습니까?,의가 있었습니까?
746,8,0.750000,제 불찰입니다.,불법입니다.
3556,26,0.730769,그러면 좀 더 많이 하지 왜 이렇게 적게 해요?,다. 그러면 조금 더 많이 하지 왜 하나 해요?
1103,16,0.709677,극단적으로 편향되고 있습니다.,적으로 많이 하고 있습니다.
745,8,0.666667,제 불찰입니다.,실정입니다.
2638,29,0.655172,"그런데 스튜디오를 다시 만들 필요가 있어요, 별도로?",그런데 별도로 스튜디오를 또 구축할 필요가 있나요?


유사도가 높은 쌍은 격식어가 겹친 짧은 인용과 같은 내용이 다른 문서에도 있는 경우다. 길이 기준을 함께 두면 짧은 인용의 우연한 겹침이 빠진다.

In [26]:
MIN_CHARS = M.check.quote_match.min_chars
for name, d in [("바꿔 옮긴 인용", bad), ("맞지 않는 대조", neg)]:
    long_ = d[d["chars"] >= MIN_CHARS]
    print(f"{name}: {MIN_CHARS}자 이상 {len(long_)}개, 유사도 최고 {long_['ratio'].max():.2f}, "
          f"최저 {long_['ratio'].min():.2f}")

바꿔 옮긴 인용: 15자 이상 14개, 유사도 최고 0.99, 최저 0.86
맞지 않는 대조: 15자 이상 4192개, 유사도 최고 0.82, 최저 0.23


In [27]:
sim = []
for _, row in ok.iterrows():
    rec = row["rec"]
    _, v = verdicts(rec, rec["elements"], json.loads(rec["response"])["answerable"],
                    M.check.quote_match)
    similar = sum(1 for el in rec["elements"] for s in el["support"]
                  if s["doc_id"] and not quoted(s["quote"], context(s["doc_id"]))
                  and found(s["quote"], context(s["doc_id"]), M.check.quote_match))
    sim.append({"pool_id": row["pool_id"], "type": row["type"], "exact": row["passed_new"],
                "similarity": v["passed"], "similar": similar,
                "dropped": v["dropped_quotes"], "reasons": v["reasons"]})
sim = pd.DataFrame(sim)
n_dep = sim["reasons"].apply(lambda r: "quote_dependent" in r).sum()
print(dict(M.check.quote_match), "| 유사도로 인정한 인용", sim["similar"].sum(),
      "| 인정하지 못한 인용", sim["dropped"].sum(), "| quote_dependent", n_dep)
sim.groupby("type")[["exact", "similarity"]].sum()

{'min_ratio': 0.8, 'min_chars': 15} | 유사도로 인정한 인용 14 | 인정하지 못한 인용 0 | quote_dependent 0


,exact,similarity
type,,
conf,5,5
law,6,6
questioner,4,4
